<a href="https://www.coursera.org/"><img src="coursera.svg" alt="coursera" width="20%"></a>
<a href="https://www.coursera.org/"><img src="./aws_logo.svg" alt="Amazon Web Services" width="5%"></a>


[Building Data Lake on AWS](https://www.coursera.org/learn/introduction-to-designing-data-lakes-in-aws/home/welcome) > [Module 2](https://www.coursera.org/learn/introduction-to-designing-data-lakes-in-aws/home/module/2) > [Exercise 1: Creating an Amazon OpenSearch Service Cluster](https://www.coursera.org/learn/introduction-to-designing-data-lakes-in-aws/supplement/RI37M/exercise-1-creating-an-amazon-opensearch-service-cluster#)

# Exercise: Creating an Amazon OpenSearch Service Cluster [#](https://aws-tc-largeobjects.s3.us-west-2.amazonaws.com/DEV-AWS-MO-Designing_DataLakes/exercise-1-es.html)

*[version_1.0]*

**Note**

The exercises in this course will have an associated charge in your AWS account. In this exercise, you will create the following resources:

- AWS Identity and Access Management (IAM) policy and user (policies and users are AWS account features, offered at no additional charge)
- Amazon Simple Storage Service (Amazon S3) bucket
- Amazon OpenSearch Service cluster
- AWS Lambda function
- Amazon API Gateway API

**The final exercise task includes instructions to delete all the resources that you create for this exercise.**

Familiarize yourself with [Amazon S3 pricing](https://aws.amazon.com/s3/pricing/), [Amazon OpenSearch Service pricing](https://aws.amazon.com/opensearch-service/pricing/), [AWS Lambda pricing](https://aws.amazon.com/lambda/pricing/), [Amazon API Gateway pricing](https://aws.amazon.com/api-gateway/pricing/), and the [AWS Free Tier](https://aws.amazon.com/free/).

## Prequest

(1) Apply AWS CDK project template app for python

```sh
    mkdir opensarch && cd opensearch
    cdk init app --language python
```


### Welcome to your CDK Python project!

This is a blank project for CDK development with Python.

The `cdk.json` file tells the CDK Toolkit how to execute your app.

This project is set up like a standard Python project.  The initialization
process also creates a virtualenv within this project, stored under the `.venv`
directory.  To create the virtualenv it assumes that there is a `python3`
(or `python` for Windows) executable in your path with access to the `venv`
package. If for any reason the automatic creation of the virtualenv fails,
you can create the virtualenv manually.

To manually create a virtualenv on MacOS and Linux:

```
$ python -m venv .venv
```

After the init process completes and the virtualenv is created, you can use the following
step to activate your virtualenv.

```
$ source .venv/bin/activate
```

If you are a Windows platform, you would activate the virtualenv like this:

```
% .venv\Scripts\activate.bat
```

Once the virtualenv is activated, you can install the required dependencies.

```
$ pip install -r requirements.txt
```

At this point you can now synthesize the CloudFormation template for this code.

```
$ cdk synth
```

To add additional dependencies, for example other CDK libraries, just add
them to your `setup.py` file and rerun the `pip install -r requirements.txt`
command.

### Useful commands

 * `cdk ls`          list all stacks in the app
 * `cdk synth`       emits the synthesized CloudFormation template
 * `cdk deploy`      deploy this stack to your default AWS account/region
 * `cdk diff`        compare deployed stack with current state
 * `cdk docs`        open CDK documentation

Enjoy!

Initializing a new git repository...\
warning: in the working copy of '.gitignore', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'README.md', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'app.py', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'cdk.json', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'requirements-dev.txt', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'requirements.txt', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'source.bat', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'test/test_stack.py', LF will be replaced by CRLF the next time Git touches it\
warning: in the working copy of 'tests/unit/test_test_stack.py', LF will be replaced by CRLF the next time Git touches it\
Please run 'python -m venv .venv'!\
Executing Creating virtualenv...\
✅ All done!\


Conten of `requirements.txt` for CDK project:
```pip-requirements
    # requirements.txt
    # This file is used to specify the dependencies for the AWS CDK project.
    aws-cdk-lib==2.180.0
    constructs>=10.0.0,<11.0.0
    aws-cdk.aws-lambda-python-alpha>=2.161.1a0
    jsii>=1.80.0,<2.0.0
    typeguard>=2.13.3,<3.0.0
    python-dotenv
    boto3
    requests
    requests-aws4auth
    pytest==6.2.5
    ipykernel
```

Conten of `requirements.txt` for lambda layer:
```pip-requirements
    # requirements.txt
    # This file is used to specify the dependencies for the lambda layer
    requests
    requests-aws4auth
```

## Setting up

In this exercise, you will ingest mock data into a data lake. Download the following .zip file that contains sample data: [`upload-data`](https://aws-tc-largeobjects.s3.us-west-2.amazonaws.com/DEV-AWS-MO-Designing_DataLakes/downloads/upload-data.zip). You will use the file for the Lambda function in the following task.

Before you begin ingesting data into a data lake, you must create an AWS Identity and Access Management (IAM) role. An IAM role defines specific account permissions, or what you can or can’t do in the AWS Cloud.

To complete instructions in this exercise, you must grant full access permissions to Amazon S3 and Amazon OpenSearch Services.

01. In the AWS Management Console menu bar, in the search box, enter IAM and then open the IAM dashboard by choosing **IAM**.
02. In the navigation pane, choose **Roles**.
03. Choose **Create role**.
04. For **Use case**, select **Lambda** and then choose **Next**.
05. In the **Permissions policies** search box, enter `AmazonS3` and press Enter.
06. From the list of results, select `AmazonS3FullAccess`.
07. Clear the **AmazonS3** filter.
08. In the search box, enter `AmazonES` and press Enter.
9. Select `AmazonESFullAccess` and then choose **Next**.
10. For Role name, paste `data-lake-week-2`.
11. Choose **Create role**.

Create IAM Role in CDK Stack

```python
        # Setting up: 
        # Create IAM role for OpenSearch
        # and assign policies: AmazonS3FullAccess and AmazonESFullAccess 
        role = iam.Role(
            self, "data-lake-week-2-role",
            role_name=f"data-lake-week-2-role-{environment}",
            description="IAM role for Lambda to access OpenSearch, S3, etc.",
            assumed_by=iam.ServicePrincipal("lambda.amazonaws.com"),
            managed_policies=[
                iam.ManagedPolicy.from_aws_managed_policy_name("service-role/AWSLambdaBasicExecutionRole"),
                iam.ManagedPolicy.from_aws_managed_policy_name("service-role/AWSLambdaVPCAccessExecutionRole"),
                iam.ManagedPolicy.from_aws_managed_policy_name("AmazonS3FullAccess"),
                iam.ManagedPolicy.from_aws_managed_policy_name("AmazonESFullAccess"),
            ], 
        )
```

### Task 1: Creating an Amazon OpenSearch Service cluster

You will use OpenSearch Service for its cataloging and indexing capabilities. Before you start ingesting documents to your OpenSearch Service domain, you must create a domain. You could also use AWS Glue for cataloging your data. However, you will explore AWS Glue in a future lesson.

In this task, you create an OpenSearch Service domain.

1. Choose **Services**, and search for and open **Amazon OpenSearch Service**.
2. Choose **Create domain** and configure the following settings.
    - **Domain name**: `water-temp-domain`
    - **Deployment type**: *Development and testing*
    - **Version**: Choose the latest version of *OpenSearch*
    - **Data nodes, Instance type**: *t3.small.search*
    - **Network**: *Public access*
    - **Enable fine-grained access control**: Clear this setting
    - **Access policy, Domain access policy**: *Configure domain level access policy*

    In the **Elements** section, you can configure a principal that is allowed to access the domain. Use your IPv4 address to restrict access to the OpenSearch Service domain.

3. First, find your IPv4 address by using an online lookup service (such as [`What Is My IP`](https://www.whatismyip.com/)) and note your **IPv4** address.
4. In the **Elements** section, configure the following settings.
    - **Type**: *IPv4 address*
    - **Principal**: Replace the asterisk (`*`) with your *IPv4 address*
    - **Action**: *Allow*
5. Choose the **JSON** tab and note that the policy only allows your IPv4 address to access the OpenSearch Service domain:
```json
    {
      "Version": "2012-10-17",
      "Statement": [
        {
          "Effect": "Allow",
          "Principal": {
            "AWS": "*"
          },
          "Action": [
            "es:*"
          ],
          "Resource": "arn:aws:es:us-east-1:000000000000:domain/water-temp-domain/*",
          "Condition": {
            "IpAddress": {
              "aws:SourceIp": [
                "x.x.x.x"
              ]
            }
          }
        }
      ]
    }
```
6. Choose **Create**.

**Note**: The domain-creation process can take up to 15 minutes to complete.

Create OpenSearch Domain in AWS CDK Stack:

```python
        # Task 1: Creating an Amazon OpenSearch Service cluster
        # Create a public OpenSearch domain
        domain_name = f"water-temp-domain-{environment}"
        resource = f"arn:aws:es:{self.region}:{self.account}:domain/{domain_name}/*"
        domain = opensearch.Domain(
            self, "WaterTempDomainSbx",
            domain_name=domain_name,
            version=opensearch.EngineVersion.OPENSEARCH_2_17,
            capacity=opensearch.CapacityConfig(
                data_node_instance_type="m7g.medium.search",
                data_nodes=3,
                master_nodes=3,
                multi_az_with_standby_enabled=True  # ✅ Correct way to enable standby
            ),
            zone_awareness=opensearch.ZoneAwarenessConfig(
                enabled=True,
                availability_zone_count=3,
            ),
            ebs=opensearch.EbsOptions(
                volume_size=10,
                volume_type=ec2.EbsDeviceVolumeType.GP3,
            ),
            removal_policy=RemovalPolicy.DESTROY, # Destroy by default, for production use RETAIN
            enforce_https=True,
            node_to_node_encryption=True,
            encryption_at_rest=opensearch.EncryptionAtRestOptions(enabled=True),
            access_policies=[   # 👇 No VPC = public access
                iam.PolicyStatement(
                    effect=iam.Effect.ALLOW,
                    principals=[iam.AnyPrincipal()],
                    actions=["es:*"],
                    resources=[resource],
                    conditions={
                        "IpAddress": {
                            "aws:SourceIp": ipv4_allowed
                        }
                    }
                ),
                # lambda permission to access OpenSearch
                iam.PolicyStatement(
                    effect=iam.Effect.ALLOW,
                    principals=[iam.ArnPrincipal(role.role_arn)],
                    actions=["es:ESHttpPut", "es:ESHttpPost", "es:ESHttpGet"],
                    resources=[resource],
                ),
            ],
            fine_grained_access_control=None,  # ❌ Not enabled
        )
```

### Task 2: Creating an S3 bucket

In this task, you create an object storage bucket for the data that’s collected from sensors.

In this exercise, you use OpenSearch Service for one specific use case. Though you will load your data into OpenSearch Service, Amazon S3 will serve as the storage layer for your data lake. The idea is that you likely have multiple use cases for the data in a data lake. Each use case uses its own set of appropriate services and tooling. By storing the raw data in Amazon S3, the data is then accessible for many use cases and various services.

To create an S3 bucket:

1. Choose **Services**, and search for and open **S3**.
2. Choose **Create bucket**.
3. Enter a bucket name.

    The bucket name must be globally unique and DNS compliant. You can name the bucket similar to the following example by using your initials for the FMI, which stands for *Fill Me In*. When you replace the FMI with your own value, make sure that you also delete the angle brackets (`<>`).

    ```sh
    datalakes-week2-<FMI> 
    ```

    **Note**: If bucket name isn’t available after you add your initials, add some numbers to the end of the name.

    Examples:

    ```sh
    datalakes-week2-sbx
    datalakes-week2-liuje-sbx
    ```

4. For **AWS Region**, the selected Region should be *US East (N. Virginia) us-east-1*.

    The bucket Region should be the same Region that your OpenSearch Service cluster is in.

5. Choose **Create bucket**.
6. Note the name of your S3 bucket. You will need to use its name in future steps.

You now have an S3 bucket that is the storage layer for your data lake. In a later task, you modify the bucket access policy so that Lambda can write files to the bucket. You will learn about bucket access policies in future lessons.

Create S3 Bucket in AWS CDK Stack:
```python
        # Task 2: Creating an S3 bucket with versioning and lifecycle rules
        # Create an S3 bucket for storing data :)
        bucket = s3.Bucket(
            self, "OpensearchBucket",
            bucket_name=bucket_name,
            versioned=True,
            removal_policy=RemovalPolicy.DESTROY,
            auto_delete_objects=True,
            lifecycle_rules=[
                s3.LifecycleRule(
                    id="ExpireOldVersions",
                    expiration=Duration.days(30),
                    noncurrent_version_expiration=Duration.days(7),
                    abort_incomplete_multipart_upload_after=Duration.days(7),
                ),
                s3.LifecycleRule(
                    id="DeleteMarkers",
                    prefix="",
                    enabled=True,
                    expired_object_delete_marker=True,
                ),
            ],
            public_read_access=False,
            block_public_access=s3.BlockPublicAccess.BLOCK_ALL,
            encryption=s3.BucketEncryption.S3_MANAGED,
            enforce_ssl=True,
            server_access_logs_prefix=f"{bucket_name}_",
            server_access_logs_bucket=s3.Bucket.from_bucket_name(
                self, 
                "AccessLogsBucket", 
                f"aws-controltower-{self.account}-{self.region}-s3-logs",
                ),
        )
        # Add bucket policy to allow OpenSearch to access the bucket
        bucket.add_to_resource_policy(
            iam.PolicyStatement(
                effect=iam.Effect.ALLOW,
                principals=[iam.ArnPrincipal(role.role_arn)],
                actions=["s3:*", ],
                resources=[f"{bucket.bucket_arn}/*"],
            )
        )
        # Acknowledge  and suppress CDK warning for access logs policy
        Annotations.of(bucket).acknowledge_warning(
            id="@aws-cdk/aws-s3:accessLogsPolicyNotAdded",
            message="Target logging bucket is imported and already has correct permissions"
        )
```

### Task 3: Creating the Lambda function

AWS Lambda is a serverless compute service. The code that you upload to a Lambda function doesn’t run continually. Instead, it runs when an event occurs.

In this task, the code for the Lambda function is already written. However, you need to create and configure the Lambda function.

The Lambda function first captures the data that is in the payload of the incoming request. Next, it uploads a JSON document to Amazon S3. Finally, the Lambda function uploads the document to OpenSearch Service.

To create the Lambda function:

01. Choose **Services**, and search for and open **Lambda**.
02. Choose **Create function** and configure the following settings.
    - **Author from scratch**: Keep this option selected
    - **Function name**: `upload-data`
    - **Runtime**: Select the *latest supported version of Python*
    - **Permissions**: Expand *Change default execution role*
    - **Execution role**: *Use an existing role*
    - **Existing role**: *data-lake-week-2*

    The Lambda function you create needs access to permissions to sign API calls that write to Amazon S3 and OpenSearch Service.

03. Choose **Create function**.
04. Scroll to the **Code** tab.
05. From the **Upload** from menu, choose **.zip file**.
06. Choose **Upload**.
07. Browse to where you saved the `upload-data.zip` file, choose the file, and choose **Open**.
08. Back in the **Upload a .zip file** dialog box, choose **Save**.
09. In the **Code** tab, scroll to **Runtime settings** and choose **Edit**.
10. In the **Handler** box, replace the existing value with `lambda.handler` and choose **Save**.
11. Choose the **Configuration** tab.
12. On the tab menu, choose **Environment variables** and then choose **Edit**.
13. Choose **Add environment variable** and configure the following settings.
    - **Key**: `S3_BUCKET`
    - **Value**: Paste your bucket name

    Example:
    
    ```yaml
    datalakes-week2-emr
    ```
14. Choose **Save**.

    You can use environment variables to adjust your function’s behavior without updating code. In a future step, you will add another environment variable for the OpenSearch Service domain.

15. In the **Configuration** tab menu, choose **Permissions**.
16. Under **Role name**, choose the **data-lake-week-2** link.

    **Note**: This action opens the IAM console in a separate window.
17. Copy the role Amazon Resource Name (ARN), which should look similar to the following example:

    ```yaml
    arn:aws:iam::000000000000:role/data-lake-week-2
    ```

You need this ARN for the following task.

Create lambda layer and lambda function in AWS CDK Stack:
```python
        # Task 3: Creating the Lambda function
        # handler is lambda.handler with environment variables:
        # AWS_BUCKET_NAME, ENVIRONMENT, ES_DOMAIN_URL
        # The Lambda function is in src/lambda.py
        # assign the role to the lambda function
        # Create lambda layer with requests and AWS4Auth, creating via bundling option
        # Define the Lambda Layer
        lambda_layer = _lambda.LayerVersion(
            self, "WaterTempLambdaLayer",
            code=_lambda.Code.from_asset("src",
                bundling=dict(
                    user="root",
                    image=_lambda.Runtime.PYTHON_3_13.bundling_image,
                    command=[
                        "bash", "-c",
                        """
                        python -m pip install --upgrade pip --root-user-action=ignore &&
                        pip install --no-cache -r requirements.txt -t /asset-output/python --root-user-action=ignore #&&
                        # unzip -qo pyodbc-mssql-lambdaplayer.zip -d /asset-output
                        """
                    ],
                ),
            ),
            compatible_runtimes=[_lambda.Runtime.PYTHON_3_13],
            layer_version_name=f"water-temp-function-{environment}-lambda-layer",
            description="Lambda Layer with Requests, etc.",
        )
        # Create the Lambda function
        lambda_function = _lambda.Function(
            self, "WaterTempFunction",
            function_name=f"water-temp-function-{environment}",
            runtime=_lambda.Runtime.PYTHON_3_13,
            handler="lambda.handler",
            code=_lambda.Code.from_asset("src"),
            environment={
                "AWS_BUCKET_NAME": bucket.bucket_name,
                "ENVIRONMENT": environment,
                "ES_DOMAIN_URL": domain.domain_endpoint,
            },
            role=role,
            memory_size=128,
            timeout=Duration.seconds(30),
            log_retention=logs.RetentionDays.ONE_MONTH,
            layers=[lambda_layer],
        )
```

### Task 4: Modifying the S3 bucket policy and OpenSearch Service cluster for Lambda access

The Lambda function writes data to the S3 bucket. You already associated an IAM role with the Lambda function in the **Setting up** section. However, the role by itself won’t allow access. You must also modify the S3 bucket policy, which determines whether AWS principals are allowed or denied access to the bucket.

01. Return to the **Amazon S3** console.
02. Open the bucket that you created previously in this exercise.
03. Choose the **Permissions** tab, scroll to **Bucket policy**, and choose **Edit**.
04. In the following JSON code, replace the first FMI with the Lambda role ARN:
    ```json
    {
        "Version": "2012-10-17",
        "Id": "ExamplePolicy",
        "Statement": [
            {
                "Sid": "ExampleStmt",
                "Effect": "Allow",
                "Principal": {
                    "AWS": "<FMI>"
                },
                "Action": "s3:*",
                "Resource": "<FMI>/*"
            }
        ]
    } 
    ```
    Example:
    ```json
            "Principal": {
                "AWS": "arn:aws:iam::xxxxxxxxxxxx:role/data-lake-week-2"
            },
    ```
05. Replace the second FMI with the ARN of your bucket:

    Example:
    ```json
            "Action": "s3:*",
            "Resource": "arn:aws:s3:::datalakes-week2-emr/*"
        }
    ```
06. In the **Bucket policy editor**, paste the bucket policy that you configured.
This policy allows the Lambda function to upload the temperature files to your bucket. The final policy should look similar to the following example, but with your account number and bucket name:
    ```json
    {
        "Version": "2012-10-17",
        "Id": "ExamplePolicy",
        "Statement": [
            {
                "Sid": "ExampleStmt",
                "Effect": "Allow",
                "Principal": {
                    "AWS": "arn:aws:iam::xxxxxxxxxxxx:role/data-lake-week-2"
                },
                "Action": "s3:*",
                "Resource": "arn:aws:s3:::datalakes-week2-xxx/*"
            }
        ]
    }
    ```
07. Choose **Save changes**.
08. Return to the **OpenSearch Service** console.
09. Choose **water-temp-domain**.

    The **Domain status** should now say “Active.” If not, wait a few minutes and refresh the page until the status is “Active.”

    **Note**: Make sure that you are in the **N. Virginia** Region.
10. Note the **Domain endpoint**, which should look similar tp the following example:
    ```txt
    https://search-water-temp-domain-xxxxxxxxxxxxxxxxxxxxxxxxx.us-east-1.es.amazonaws.com
    ```
11. Choose **Actions** and select **Edit security configuration**.
12. Scroll to **Access policy** and review the policy.

    Currently, the policy restricts access by IP address. The current policy should look similar to the following example:

    Example:
    ```json
    {
    "Version": "2012-10-17",
    "Statement": [
        {
        "Effect": "Allow",
        "Principal": {
            "AWS": "*"
        },
        "Action": "es:*",
        "Resource": "arn:aws:es:us-east-1:000000000000:domain/water-temp-domain/*",
        "Condition": {
            "IpAddress": {
            "aws:SourceIp": "x.x.x.x"
            }
        }
        }
    ]
    }
    ```
    The Lambda function should also have access to the domain, so you will add it in the following step.

13. Add Lambda access to the policy and update values with your account number.

    The policy must look similar to the following example, but with your account number and IP address:

    ```json
    {
    "Version": "2012-10-17",
    "Statement": [
        {
        "Effect": "Allow",
        "Principal": {
            "AWS": "*"
        },
        "Action": "es:*",
        "Resource": "arn:aws:es:us-east-1:000000000000:domain/water-temp-domain/*",
        "Condition": {
            "IpAddress": {
            "aws:SourceIp": "x.x.x.x"
            }
        }
        },
        {
        "Effect": "Allow",
        "Principal": {
            "AWS": "arn:aws:iam::000000000000:role/data-lake-week-2"
        },
        "Action": "es:*",
        "Resource": "arn:aws:es:us-east-1:000000000000:domain/water-temp-domain/*"
        }
    ]
    }
    ```
14. Choose **Save changes**.


### Task 5: Modifying the Lambda function and creating an API Gateway endpoint

You now have an OpenSearch Service domain and an S3 bucket. You need to modify one final thing for the Lambda function to work.

After the Lambda function is configured, you need a way to run the function. Lambda functions run based on events. In this case, the sensors that record the temperature data make HTTPS POST requests, with the data in the payload. You use API Gateway to create an API endpoint that receives the HTTPS POST requests. After API Gateway receives and validates the request, it invokes the Lambda function and passes the information to it. The Lambda function then writes the data to Amazon S3 and OpenSearch Service.

#### Step 5.1: Adding a new environment variable to Lambda

01. Return to the **Lambda** console.
02. Choose the **upload-data** function and then choose the **Configuration** tab.
03. On the tab menu, choose **Environment variables** and then choose **Edit**.
04. Choose **Add environment variable** and configure the following settings.
  
    - **Key**: `ES_DOMAIN_URL`
    - **Value**: Paste the OpenSearch Service domain endpoint that you noted previously.

    It should look similar to the following example:
    ```txt
    https://search-water-temp-domain-xxxxxxxxxxxxxxxxxxxxxxxxx.us-east-1.es.amazonaws.com
    ```
    Ensure that the OpenSearch Service domain endpoint doesn’t have a slash (/) at the end.

05. Choose **Save**.

#### Step 5.2: Creating a REST API

After you set up the variable for OpenSearch Service, you create a REST API in Amazon API Gateway to receive data from the sensors. In this exercise, you manually enter test data to simulate requests from the sensors.

01. Choose **Services**, and search for and open **API Gateway**.

02. On the **REST API** card, choose **Build**.
03. If needed, close the **Create your first API** dialog box by choosing **OK**.
04. For **Create new API**, select **New API**.
05. In the **API name** box, paste s`ensor-data` and choose **Create API**.
06. In the **Resources** pane, on the **Actions** menu, choose **Create Method**.
07. On the dropdown menu, choose **POST**, and confirm your selection by choosing the checkmark.
08. In the **POST - Setup** pane, keep the **Integration type** setting at **Lambda Function**.
09. In the **Lambda Function** box, enter ``upload-data`` and select it when it appears.
10. Choose **Save**, and in the **Add Permission to Lambda Function** dialog box, choose **OK**.
11. In the **POST - Method Execution pane**, choose **TEST**.
12. Scroll to the **Request Body** box and paste the following JSON code:

  ```json
      {
      "sensorID" : "0025",
      "temperature" : "67"
      }
  ```
13. Choose **Test**. In the **Response Body** section, you should see a response similar to the following example:

    ```json
    {
      "Response": "Data Uploaded",
      "sensorID": "0025",
      "temperature": "67"
    }
    ```

    This test simulates the POST request that would come in from different IoT sensors.